# 63 — Regime debole (QCA9880, 5 GHz): modelli classici e quantistici a parità di rischio e di budget

Il regime è quello individuato nel notebook 62: jammer a **60 cm**, soprattutto a **0 dBm**, su canali a 5 GHz.

**Disegno.** Lo stesso del notebook 62, nella stessa campagna:
- per ogni canale f a 5 GHz, **negativi** = finestre a f dei file con il jammer a 2,4 GHz; **positivi** = finestre a f dei file con il jammer su f;
- fino a 5 finestre non sovrapposte per file e canale, per avere più dati;
- split **per file** 50/25/25;
- le feature sono **standardizzate per canale** sui negativi di training, poi tutti i canali vengono messi insieme: un solo modello per la banda.

**Modelli, a parità di input** (7 medie per finestra; per alcuni modelli anche le covarianze):
- **one-class**, addestrati sui soli negativi:
  - Mahalanobis, residuo PCA, Isolation Forest, One-Class SVM RBF, RFF-Mahalanobis;
  - One-Class SVM con **kernel quantistico** (7 qubit, RY, anello di CNOT, re-uploading), **controllo separabile** e RBF sugli stessi angoli;
- **supervisionati**, per riferimento:
  - Random Forest, regressione logistica;
  - **MLP piccolo** (~91 parametri) contro **VQE-inspired a 7 qubit** con lettura affine (44 parametri), 3 seed ciascuno.

**Metriche:**
- AUROC con intervallo di DeLong;
- rilevamento a α = 1% e 0,1% con **soglia conforme** sui negativi di calibrazione, e falso allarme osservato;
- test di DeLong tra il miglior modello quantistico e il miglior classico;
- tutto per W = 8, 32, 128 e per distanza × potenza.

In [ ]:
import os, sys, time, pathlib, json
REPO = pathlib.Path.cwd()
while not (REPO / 'qdetect').exists() and REPO != REPO.parent: REPO = REPO.parent
sys.path.insert(0, str(REPO))
import numpy as np, pandas as pd
from multiprocessing import Pool
from sklearn.svm import OneClassSVM
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics.pairwise import rbf_kernel
import qdetect as qd
from qdetect import passive as P, oneclass as O, qkernel as QK, quantum as Q
pd.set_option('display.width', 230); pd.set_option('display.max_rows', 400)
BASE = pathlib.Path(os.environ.get('QCA_BASE', '/home/habes/NotebookProjects/bboi/monaco'))
ROOT = BASE / 'spectral_scans_QCA9880'
N_JOBS = int(os.environ.get('N_JOBS', '16'))
SEEDS = [int(s) for s in os.environ.get('SEEDS', '42,123,2024').split(',')]
VQE_STEPS = int(os.environ.get('VQE_STEPS', '600'))
OUT = REPO / 'results' / 'regime_debole'; OUT.mkdir(parents=True, exist_ok=True)
inv = pd.read_csv(BASE / 'inventory_full.csv')
inv = inv[(inv.is_processed == True) & (inv.band == 'ht20') & (inv.environment == 'chamber') & inv.freq_mhz.notna()].copy()
CH5 = sorted(int(f) for f in inv.freq_mhz.unique() if f > 3000)
print(len(inv), 'file di jamming;', len(CH5), 'canali a 5 GHz:', CH5)
WINDOWS = (8, 32, 128); MEANS = [f'm_{k}' for k in P.ALL7]
SECOND = P.feature_columns(P.ALL7, 'second')

## 1. Finestre (fino a 5 per file e canale) ed etichette

In [ ]:
def work(rel):
    d = pd.read_csv(ROOT / rel)
    return [dict(r, full_path=rel) for r in P.channel_windows(d, P.ALL7, WINDOWS, max_windows=5, channels=CH5)]
t0 = time.time()
with Pool(N_JOBS) as pool:
    recs = [r for part in pool.imap_unordered(work, inv.full_path.tolist(), chunksize=16) for r in part]
F = pd.DataFrame(recs).merge(inv[['full_path', 'freq_mhz', 'distance_cm', 'jam_power_dbm']], on='full_path')
F['y'] = np.where(F.freq_mhz == F.channel, 1, np.where(F.freq_mhz < 3000, 0, -1))
F = F[F.y >= 0].copy()                                   # esclusi i file con jammer su un altro canale a 5 GHz
F['split'] = F.full_path.map(dict(zip(inv.full_path, P.file_split(inv.full_path.tolist()))))
print(f'{len(F)} finestre in {time.time()-t0:.0f} s'); print(F.groupby(['W', 'split', 'y']).size().unstack())

## 2. Standardizzazione per canale (sui negativi di training) e insiemi

In [ ]:
def prepare(W, cols):
    G = F[F.W == W].copy()
    ref = G[(G.y == 0) & (G.split == 'train')].groupby('channel')[cols].agg(['mean', 'std'])
    Z = np.empty((len(G), len(cols)))
    for c, idx in G.groupby('channel').indices.items():
        mu = ref.loc[c].xs('mean', level=1)[cols].to_numpy(float); sd = ref.loc[c].xs('std', level=1)[cols].to_numpy(float) + 1e-6
        Z[idx] = np.clip((G.iloc[idx][cols].to_numpy(float) - mu) / sd, -50, 50)
    return G.reset_index(drop=True), Z
def subsets(G):
    return {k: G.index[m].to_numpy() for k, m in {
        'tr_neg': (G.y == 0) & (G.split == 'train'), 'cal_neg': (G.y == 0) & (G.split == 'cal'),
        'te_neg': (G.y == 0) & (G.split == 'test'), 'tr_pos': (G.y == 1) & (G.split == 'train'),
        'cal_pos': (G.y == 1) & (G.split == 'cal'), 'te_pos': (G.y == 1) & (G.split == 'test')}.items()}
rng = np.random.default_rng(0)
def cap(idx, n): return idx if len(idx) <= n else np.sort(rng.choice(idx, n, replace=False))

## 3. Modelli e valutazione

In [ ]:
rows, scores = [], {}
CONFIGS = [(60, 0), (60, 5), (60, 10), (40, 0)]
def evaluate(name, family, W, rep, s_cal, s_neg, s_pos_all, G, I, seed=None, n_params=None, train_s=None):
    scores[(W, rep, name, seed)] = (s_neg, s_pos_all)
    pos_meta = G.iloc[I['te_pos']]
    for (dist, pw) in CONFIGS + [('tutte', 'tutte')]:
        m = np.ones(len(pos_meta), bool) if dist == 'tutte' else ((pos_meta.distance_cm == dist) & (pos_meta.jam_power_dbm == pw)).to_numpy()
        sp = s_pos_all[m]; y = np.r_[np.zeros(len(s_neg)), np.ones(len(sp))]
        auc, lo, hi = qd.auroc_ci(y, np.r_[s_neg, sp])
        r = dict(W=W, rep=rep, model=name, family=family, seed=seed, distance_cm=dist, power_dbm=pw, n_neg=len(s_neg), n_pos=len(sp),
                 n_pos_files=pos_meta[m].full_path.nunique(), auroc=auc, auroc_lo=lo, auroc_hi=hi, n_params=n_params, train_s=train_s)
        for a in (0.01, 0.001):
            h = O.conformal_threshold(s_cal, a); fp = int((s_neg > h).sum())
            r[f'tpr@{a:g}'] = float((sp > h).mean()); r[f'fpr@{a:g}'] = fp / len(s_neg); r[f'fpr_up95@{a:g}'] = O.fpr_upper95(fp, len(s_neg))
        rows.append(r)

t_all = time.time()
for W in WINDOWS:
    for rep, cols in [('mean', MEANS), ('mean+cov', SECOND)]:
        G, Z = prepare(W, cols); I = subsets(G)
        trn = cap(I['tr_neg'], 30000)
        # --- one-class classici
        for name, fn in O.one_class_models(Z[trn], seed=0):
            t0 = time.perf_counter(); evaluate(name, 'classico one-class', W, rep, fn(Z[I['cal_neg']]), fn(Z[I['te_neg']]), fn(Z[I['te_pos']]), G, I, train_s=time.perf_counter()-t0)
        if rep != 'mean':
            continue
        # --- kernel quantistico (7 qubit) e controlli sugli stessi angoli, One-Class SVM precalcolato
        A = np.pi * np.tanh(Z / 2); ocn = cap(I['tr_neg'], 1500)
        for kname, fam, kf in [('ocsvm kernel quantistico', 'quantistico one-class', lambda X, Y: QK.fidelity_kernel(X, Y, True)),
                               ('ocsvm kernel separabile', 'quantistico one-class', lambda X, Y: QK.fidelity_kernel(X, Y, False)),
                               ('ocsvm rbf angoli', 'classico one-class', lambda X, Y: rbf_kernel(X, Y, gamma=1 / 7))]:
            t0 = time.perf_counter(); oc = OneClassSVM(kernel='precomputed', nu=0.05).fit(kf(A[ocn], A[ocn]))
            sc = lambda idx: np.concatenate([-oc.decision_function(kf(A[idx[i:i+4000]], A[ocn])) for i in range(0, len(idx), 4000)])
            evaluate(kname, fam, W, rep, sc(I['cal_neg']), sc(I['te_neg']), sc(I['te_pos']), G, I, train_s=time.perf_counter()-t0)
        # --- supervisionati (negativi + positivi di training, tutte le distanze e potenze)
        sup = np.r_[cap(I['tr_neg'], 20000), I['tr_pos']]; ys = G.y.to_numpy()[sup]
        name, fn = O.supervised_reference(Z[sup], ys, 0)
        evaluate('random forest', 'classico supervisionato', W, rep, fn(Z[I['cal_neg']]), fn(Z[I['te_neg']]), fn(Z[I['te_pos']]), G, I)
        lr = LogisticRegression(max_iter=3000).fit(Z[sup], ys)
        evaluate('regressione logistica', 'classico supervisionato', W, rep, lr.predict_proba(Z[I['cal_neg']])[:, 1], lr.predict_proba(Z[I['te_neg']])[:, 1], lr.predict_proba(Z[I['te_pos']])[:, 1], G, I, n_params=8)
        val = np.r_[cap(I['cal_neg'], 3000), I['cal_pos']]; yv = G.y.to_numpy()[val]
        for seed in SEEDS:
            t0 = time.perf_counter(); mlp = MLPClassifier(hidden_layer_sizes=(10,), max_iter=2000, early_stopping=True, random_state=seed).fit(Z[sup], ys)
            evaluate('MLP piccolo (91 par.)', 'classico supervisionato', W, rep, mlp.predict_proba(Z[I['cal_neg']])[:, 1], mlp.predict_proba(Z[I['te_neg']])[:, 1], mlp.predict_proba(Z[I['te_pos']])[:, 1], G, I, seed=seed, n_params=91, train_s=time.perf_counter()-t0)
            circ = Q.vqe_sel(7, 2)
            r = Q.train(circ, A[sup], ys, A[val], yv, readout='affine', optimizer='adam', steps=VQE_STEPS, batch=32, lr=0.03, eval_every=50, patience=6, init_scale=0.1, seed=seed)
            ps = lambda idx: Q.predict_score(r.params, circ, A[idx], 'affine')
            evaluate('VQE-inspired 7 qubit (44 par.)', 'quantistico supervisionato', W, rep, ps(I['cal_neg']), ps(I['te_neg']), ps(I['te_pos']), G, I, seed=seed, n_params=Q.n_trainable(circ, 'affine'), train_s=r.train_seconds)
        print(f'W={W} {rep}: {time.time()-t_all:.0f} s', flush=True)
R = pd.DataFrame(rows); R.to_csv(OUT / 'metriche.csv', index=False)
print(f'totale {time.time()-t_all:.0f} s, {len(R)} righe')

## 4. Risultati nel regime più difficile (60 cm, 0 dBm)

In [ ]:
K = R[R.distance_cm.astype(str).isin(['60', '60.0']) & R.power_dbm.astype(str).isin(['0', '0.0'])]
agg = K.groupby(['W', 'rep', 'family', 'model']).agg(auroc=('auroc', 'mean'), auroc_lo=('auroc_lo', 'mean'), auroc_hi=('auroc_hi', 'mean'),
       tpr1=('tpr@0.01', 'mean'), fpr1=('fpr@0.01', 'mean'), tpr01=('tpr@0.001', 'mean'), fpr01=('fpr@0.001', 'mean'),
       n_pos=('n_pos', 'first'), n_pos_files=('n_pos_files', 'first'), n_neg=('n_neg', 'first')).reset_index()
for W in WINDOWS:
    print(f'\n=== W = {W}, 60 cm, 0 dBm ==='); display(agg[agg.W == W].sort_values('auroc', ascending=False).round(3))
agg.to_csv(OUT / 'riepilogo_60cm_0dBm.csv', index=False)

## 5. Effetto di distanza, potenza e budget (miglior modello di ogni famiglia)

In [ ]:
B = R[R.rep == 'mean'].groupby(['family', 'W', 'distance_cm', 'power_dbm', 'model'])['auroc'].mean().reset_index()
best = B.sort_values('auroc').groupby(['family', 'W', 'distance_cm', 'power_dbm']).tail(1)
display(best.pivot_table(index=['distance_cm', 'power_dbm', 'W'], columns='family', values='auroc').round(3))
best.to_csv(OUT / 'migliori_per_famiglia.csv', index=False)

## 6. Test di DeLong: miglior quantistico contro miglior classico (60 cm, 0 dBm, stesse finestre)

In [ ]:
dl = []
for W in WINDOWS:
    for fam_q, fam_c in [('quantistico one-class', 'classico one-class'), ('quantistico supervisionato', 'classico supervisionato')]:
        a = agg[(agg.W == W) & (agg.rep == 'mean')]
        qn = a[a.family == fam_q].sort_values('auroc').model.iloc[-1]; cn = a[a.family == fam_c].sort_values('auroc').model.iloc[-1]
        kq = [k for k in scores if k[0] == W and k[1] == 'mean' and k[2] == qn][0]; kc = [k for k in scores if k[0] == W and k[1] == 'mean' and k[2] == cn][0]
        G, _ = prepare(W, MEANS); I = subsets(G); pm = G.iloc[I['te_pos']]
        m = ((pm.distance_cm == 60) & (pm.jam_power_dbm == 0)).to_numpy()
        (nq, pq), (nc, pc) = scores[kq], scores[kc]
        y = np.r_[np.zeros(len(nq)), np.ones(m.sum())]
        dl.append(dict(W=W, confronto=f'{qn} vs {cn}', **qd.delong_test(y, np.r_[nq, pq[m]], np.r_[nc, pc[m]])))
D = pd.DataFrame(dl); D.to_csv(OUT / 'delong_60cm_0dBm.csv', index=False); display(D.round(4))